# Testing

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*. Original material (2020) in collaboration with
*G. Martínez-Lema* and *M. Kekic*.

**Objectives**

* decide **what** to test before writing the code;
* write tests that run by themselves with **pytest**;
* test numbers with **tolerances**, and errors with `pytest.raises`;
* let **hypothesis** look for the cases you did not think of.

In [ ]:
import math
import random
import numpy as np
import pytest

## 1. Why test

In [Functions](./about_functions.ipynb), `fib(1)` returned two numbers. The code looked
right, ran without errors, and gave a plausible output. Only a test caught it.

* Code that runs is not code that is **correct**.
* A test is a **claim** about the code, written so that the computer checks it.
* Tests are run **every time** the code changes: they protect you from yourself.

### 1.1 Tests define the function

The tests say what a function must do, better than its docstring. They can be written
**before** the function.

In [ ]:
def check(f, a, b, zero):
    """Properties that a two-argument function f must have."""
    assert f(a, b) == f(b, a)
    assert f(a, f(a, b)) == f(f(a, a), b)
    assert f(a, zero) == a

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Which function?</b></p>

Read the three asserts of `check`. **Propose** a function `f` that passes them, and the
`zero` that goes with it. Is there more than one?

</div>

<details>
<summary><b>Solution</b></summary>

The properties are **commutativity**, a case of **associativity**, and a **neutral
element**. The sum with `zero = 0` passes; so does the product with `zero = 1`, `max` with
`zero = -math.inf`, or `math.gcd`
with `zero = 0` (for $a \ge 0$). The tests do not define *one* function:
they define a *family*. To pin down the sum you need more tests, e.g. `f(1, 1) == 2`.

</details>

In [ ]:
add = lambda a, b: a + b

check(add, 1 + 2j, 3 - 1j, 0)
print('complex: passed')

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — The same + on lists</b></p>

`+` also works on lists. Does `check(add, [1, 2], [3], [])` pass? Predict, then run it.

</div>

In [ ]:
check(add, [1, 2], [3], [])

<details>
<summary><b>Solution</b></summary>

It fails at the first assert: `[1, 2] + [3]` is `[1, 2, 3]` and `[3] + [1, 2]` is
`[3, 1, 2]`. On lists `+` means *concatenate*, which is not commutative. The same symbol,
different properties: the tests tell them apart.

</details>

## 2. Writing tests

A **test** is a function whose name starts with `test_`, with no arguments, that uses
`assert`. It passes if it runs to the end; it fails at the first false `assert`.

In [ ]:
def mean(xs):
    """Arithmetic mean of a non-empty sequence of numbers."""
    return sum(xs) / len(xs)

def test_mean_simple():
    assert mean([1, 2, 3]) == 2

def test_mean_one_value():
    assert mean([5.]) == 5.

test_mean_simple()
test_mean_one_value()

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — What would you test?</b></p>

Before going on, **write down** at least four more tests for `mean`. Think of special
inputs, of properties, and of what should happen with bad inputs.

</div>

<details>
<summary><b>Solution</b></summary>

Some ideas, by type:

* **special inputs**: negative numbers; a mix of `int` and `float`; numpy arrays;
* **properties**: `mean(xs)` lies between `min(xs)` and `max(xs)`; the mean of a constant
  list is the constant; `mean([x + c for x in xs]) == mean(xs) + c`; the order does not
  matter;
* **bad inputs**: `mean([])` must fail with a clear error (now it gives
  `ZeroDivisionError`: is that clear enough?);
* **a reference**: compare with `np.mean` or `statistics.mean`.

</details>

### 2.1 Floats and ==

In [ ]:
print(0.1 + 0.2 == 0.3)
print(mean([0.1, 0.1, 0.1]))

Floats are rounded. Compare with a **tolerance**: `math.isclose(a, b)` (default relative
tolerance $10^{-9}$, and it needs `abs_tol` to compare with 0), `np.allclose(xs, ys)`,
or in pytest `a == pytest.approx(b)` (default relative tolerance $10^{-6}$).

In [ ]:
def test_mean_floats():
    assert mean([0.1, 0.1, 0.1]) == pytest.approx(0.1)
    assert math.isclose(mean([1e-3, 2e-3]), 1.5e-3, rel_tol=1e-12)

test_mean_floats()

The tolerance is **physics**, not a detail: it should come from what you know about the
error of the computation (rounding, truncation, statistics).

### 2.2 Testing errors

A function must also **fail well**. `pytest.raises` checks that an error is raised:

In [ ]:
def safe_mean(xs):
    """Arithmetic mean; raises ValueError on an empty sequence."""
    if len(xs) == 0:
        raise ValueError('mean of an empty sequence')
    return sum(xs) / len(xs)

def test_mean_empty():
    with pytest.raises(ValueError):
        safe_mean([])

test_mean_empty()

### 2.3 A test must be able to fail

<div class="admonition hint">
<p class="admonition-title"><b>[?] What do you see? — Is this test useful?</b></p>

This test is meant to check that `fib` gives non-negative numbers. What is wrong with it?

```python
def test_fib_positive():
    for x in fib(0):
        assert x >= 0
```

</div>

<details>
<summary><b>Solution</b></summary>

`fib(0)` is empty (with the fixed `fib` of the Functions chapter): the loop does nothing and the
test passes without checking anything. A test must **fail** when the code is wrong. A
good habit: break the code on purpose and check that the test catches it.

</details>

## 3. pytest

Tests live in their own files, next to the code. **pytest** finds the files named
`test_*.py`, runs every `test_*` function in them, and reports.

We write a small module and its tests from the notebook (`%%writefile` saves the cell as
a file). `fca_stats.mean` is our `safe_mean`:

In [ ]:
%%writefile fca_stats.py
"""Simple statistics."""


def mean(xs):
    """Arithmetic mean of a non-empty sequence of numbers."""
    if len(xs) == 0:
        raise ValueError('mean of an empty sequence')
    return sum(xs) / len(xs)


def variance(xs):
    """Population variance of a non-empty sequence of numbers."""
    m = mean(xs)
    return sum((x - m) ** 2 for x in xs) / len(xs)

In [ ]:
%%writefile test_fca_stats.py
import numpy as np
import pytest

from fca_stats import mean, variance


def test_mean():
    assert mean([1, 2, 3]) == 2


def test_mean_empty():
    with pytest.raises(ValueError):
        mean([])


@pytest.mark.parametrize('xs', [[1.], [1., 2.], [3., -1., 4., 1.5]])
def test_variance_numpy(xs):
    assert variance(xs) == pytest.approx(np.var(xs))

In [ ]:
!pytest -q --color=no test_fca_stats.py

* `@pytest.mark.parametrize` (a decorator!) runs the same test for several inputs.
* In a terminal: `pytest` alone runs all the tests of the folder; `pytest -x` stops at the
  first failure; `pytest -k variance` runs only the tests whose name matches.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Break it</b></p>

1. Introduce a bug in `variance` in `fca_stats.py` (do not re-run the `%%writefile`
   cell, it would overwrite your change; for example, divide by
   `len(xs) - 1`) and run pytest again. Which test catches it? Read the report.
2. Add a test for a property of the variance: it does not change if you add a constant
   to all the data. Run pytest.
3. Fix the bug.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def test_variance_shift():
    xs = [3., -1., 4., 1.5]
    assert variance([x + 10. for x in xs]) == pytest.approx(variance(xs))
```

With `len(xs) - 1`, `test_variance_numpy` fails for every input with more than one value,
and the input `[1.]` fails with a `ZeroDivisionError`: the report shows the input, the
expected value and the obtained one. `test_variance_shift` still passes, because the
property holds for both definitions: **a property alone is not enough**, you need a
reference too.

</details>

***[+] Lookout.*** pytest has much more: **fixtures** (`@pytest.fixture`, to prepare data shared by
several tests), `tmp_path` (a temporary folder), markers to skip slow tests, and
`pytest --cov` (with `pytest-cov`) to see which lines of your code the tests never run.
See the [pytest documentation](https://docs.pytest.org/).

## 4. hypothesis: let the computer find the cases

You choose the inputs of your tests, so they only check the cases you thought of.
**hypothesis** generates hundreds of inputs, looking for the ones that break a
**property**, and when it finds one it **shrinks** it to a minimal failing example.

In [ ]:
from hypothesis import given, strategies as st

@given(st.lists(st.integers(-10**6, 10**6), min_size=1))
def test_mean_bounds_int(xs):
    assert min(xs) <= mean(xs) <= max(xs)

test_mean_bounds_int()
print('passed')

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — And with floats?</b></p>

The same property, with lists of any finite floats instead of integers. **Decide**: does
it pass? If not, what kind of input will hypothesis find?

</div>

In [ ]:
@given(st.lists(st.floats(allow_nan=False, allow_infinity=False), min_size=1))
def test_mean_bounds(xs):
    assert min(xs) <= mean(xs) <= max(xs)

test_mean_bounds()

<details>
<summary><b>Solution</b></summary>

It fails. Depending on the run, hypothesis finds one of two problems, both real:

* **overflow**: two numbers close to the largest float, `1.79e308`, add up to `inf`, and
  the mean is `inf`;
* **rounding**: `mean([0.1, 0.1, 0.1])` is `0.10000000000000002`, larger than the
  maximum.

Neither would appear in the tests we wrote by hand. What to do depends on the physics:
restrict the inputs to the range you will really use
(`st.floats(-1e6, 1e6)`), compare with a tolerance, or use an exactly rounded mean
(`statistics.mean`, slower). `math.fsum` only reduces the accumulation error: it does
not fix these two cases.

(Even the integer version above needed limits: beyond $2^{53}$ an `int` divided by an
`int` is rounded, and the property fails too.)

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Test a class you did not write</b></p>

The file [rational.py](./rational.py) has a class `Rational(p, q)` for the fraction
$p/q$. You do not need to know how it is written to test it: use it as you use
`complex`.

```python
from rational import Rational
a, b = Rational(1, 2), Rational(1, 3)
print(a + b, a * b, -a)
```

With hypothesis, test that the sum is commutative and associative, that `Rational(0, 1)`
is its neutral element, and compare the results with `fractions.Fraction` from the
standard library, used as a **reference**. (`Rational + int` is not supported.)

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
from fractions import Fraction
from hypothesis import given, strategies as st
from rational import Rational

pairs = st.tuples(st.integers(), st.integers().filter(lambda q: q != 0))

@given(pairs, pairs)
def test_add_like_fraction(r1, r2):
    s = Rational(*r1) + Rational(*r2)
    f = Fraction(*r1) + Fraction(*r2)
    assert (s.p, s.q) == (f.numerator, f.denominator)

@given(pairs, pairs, pairs)
def test_add_associative(r1, r2, r3):
    a, b, c = Rational(*r1), Rational(*r2), Rational(*r3)
    assert (a + b) + c == a + (b + c)

@given(pairs, pairs)
def test_add_commutative(r1, r2):
    a, b = Rational(*r1), Rational(*r2)
    assert a + b == b + a

@given(pairs)
def test_add_neutral(r):
    a = Rational(*r)
    assert a + Rational(0, 1) == a

test_add_like_fraction()
test_add_associative()
test_add_commutative()
test_add_neutral()
```

Comparing with a trusted reference checks everything at once (value, sign, reduced
form). The file [rational_test.py](./rational_test.py) has more tests of this kind.

</details>

## 5. Testing physics code

Numerical code is harder to test: the "right answer" is often unknown, and the result has
an error. Some strategies:

* **known cases**: an analytical solution, a limit ($v \ll c$, $\hbar \to 0$);
* **conservation laws and symmetries**: energy, momentum, charge; invariance under
  rotations;
* **convergence**: halve the step and check that the error goes down as expected
  ($h^2$ for a central difference);
* **a reference**: another implementation (numpy, scipy, a published table).

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — How do you test a Monte Carlo?</b></p>

A function returns $\pi$ estimated with $n$ random points in a square. Its output is
different at every run. How would you test it? What is a sensible tolerance? What role
does the random **seed** play?

</div>

<details>
<summary><b>Solution</b></summary>

* Fix the seed: the result is reproducible, and a change in the code that changes the
  result is detected (a *regression* test). But it does not say that the result is right.
* Test the **statistics**: the estimator has an error $\sigma/\sqrt{n}$; check that the
  result is within, say, 5 $\sigma$ of $\pi$. With a random seed, a $5\sigma$ test fails by
  chance less than once in a million runs; with a fixed seed it is deterministic. Choose
  the seed once: do not search for one that passes.
* Test that the error **decreases** as $1/\sqrt{n}$, averaging $|\hat\pi - \pi|$ over many
  seeds.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[&] Collaborate — Cross the tests</b></p>

In pairs. Each one writes, without looking at the other:

* a function `pi_mc(n, seed)` that estimates $\pi$ with $n$ random points;
* a set of tests for **that** function, following the discussion above.

Then run your tests on your partner's function, and theirs on yours. Did anybody's code
fail somebody else's tests? Why? This is step 5 of the [AI cycle](./python_intro.ipynb):
the same exercise works with an AI as partner.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def pi_mc(n, seed=None):
    """Estimate pi with n random points in the unit square."""
    rng = random.Random(seed)
    inside = sum(1 for _ in range(n) if rng.random() ** 2 + rng.random() ** 2 < 1.)
    return 4. * inside / n

def test_pi_reproducible():
    assert pi_mc(1000, seed=1) == pi_mc(1000, seed=1)

def test_pi_statistics():
    n = 100_000
    p = math.pi / 4
    sigma = 4. * math.sqrt(p * (1 - p) / n)
    assert abs(pi_mc(n, seed=2) - math.pi) < 5 * sigma

test_pi_reproducible()
test_pi_statistics()
```

</details>

## 6. Summary

* Test **properties** and **reference** values, not only examples.
* Floats: a tolerance. Errors: `pytest.raises`.
* Random code: a seed, and statistics.
* A test must be able to **fail**.

<details>
<summary><b>[i] Cheat sheet — Testing</b></summary>

| | |
|:--|:--|
| `assert cond, 'msg'` | the basic check |
| `def test_xxx():` in `test_*.py` | what pytest finds and runs |
| `pytest`, `pytest -x`, `pytest -k name` | run all, stop at the first failure, select |
| `x == pytest.approx(y)`, `math.isclose` | compare floats with a tolerance |
| `with pytest.raises(ValueError):` | check that an error is raised |
| `@pytest.mark.parametrize('x', [...])` | the same test for several inputs |
| `@given(st.integers())` | hypothesis: property-based tests |

**What to test**: special inputs (empty, zero, negative, huge), properties, bad inputs,
a reference. **A test must be able to fail**: break the code and check.

</details>

### Check yourself

1. Why should you not compare two floats with `==`? What do you use instead?
2. What does pytest need to find a test?
3. What is the difference between testing an example and testing a property?
4. A test passes. Does that prove the code is correct?
5. How do you test a function whose output is random?

<details>
<summary><b>Solutions</b></summary>

1. Floats are rounded: `0.1 + 0.2 != 0.3`. Use a tolerance: `math.isclose`,
   `pytest.approx`, `np.allclose`.
2. A file `test_*.py` with functions `test_*`, without arguments.
3. An example checks one input and its known output; a property must hold for every
   input (hypothesis generates them).
4. No: it only proves that the code does what the test checks, and the test may check
   nothing (an empty loop) or the wrong thing.
5. Fix the seed for reproducibility, and test the statistics: the result must be within a
   few standard deviations of the expected value.

</details>

### Bibliography

1. [pytest documentation](https://docs.pytest.org/).
2. [hypothesis documentation](https://hypothesis.readthedocs.io/).